# 2.7 Using Segmentation to Find Suspected Nodules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emreaslan7/ai/blob/main/notebooks/deep-learning-with-pytorch/15-using-segmentation-to-find-suspected-nodules.ipynb)

This notebook implements **Section 2.7: Step 2 of the CAD Pipeline** from the *Deep Learning with PyTorch (2nd Edition)* curriculum (Section 2.7).
We explore:
1. **Volumetric Slicing & HU Normalization:** Slicing 3D CT volumes into 2D axial planes.
2. **Prompt-Guided Mask Synthesis (SAM Concept):** Point-prompted ground-truth mask generation.
3. **Autonomous Segmentation (SegFormer / MLP-Decoder):** Parameter-efficient transfer learning with frozen backbone.
4. **Hybrid Dice + BCE Loss:** Overcoming the 99.97% background class imbalance.
5. **Candidate Coordinate Extraction:** Converting dense predicted probability maps into discrete 3D nodule candidates.

In [ ]:
# Cell 0: Core Setup, Imports & Hardware Device Verification
import math
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from scipy import ndimage

# Set seeds for reproducible execution
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__} | Active Device: {device}')

## 1. 3D Volumetric CT Slicing & Radiodensity Normalization

We synthesize a calibrated 2D axial thoracic CT slice (512x512 voxels) containing anatomical lung cavities (-1000 HU to -600 HU), soft mediastinal tissues (+40 HU), cortical bone (+700 HU), and a solitary pulmonary nodule centered at coordinate (row=366, col=316).

In [ ]:
def create_synthetic_ct_slice(size=512, nodule_row=366, nodule_col=316, nodule_radius=14):
    """Generates a realistic calibrated axial CT slice in Hounsfield Units."""
    y, x = np.ogrid[:size, :size]
    cy, cx = size // 2, size // 2
    
    # Background air (-1000 HU)
    slice_hu = np.full((size, size), -1000.0, dtype=np.float32)
    
    # Outer body thoracic contour (+40 HU soft tissue)
    body_mask = ((x - cx)**2 / (210**2) + (y - cy)**2 / (180**2)) <= 1.0
    slice_hu[body_mask] = 40.0
    
    # Ribs and cortical bone (+700 HU)
    bone_ring = body_mask & (((x - cx)**2 / (205**2) + (y - cy)**2 / (175**2)) >= 0.92)
    slice_hu[bone_ring] = 700.0
    
    # Left and Right Lung Cavities (-750 HU air parenchyma)
    left_lung = ((x - (cx - 90))**2 / (75**2) + (y - cy)**2 / (120**2)) <= 1.0
    right_lung = ((x - (cx + 90))**2 / (75**2) + (y - cy)**2 / (120**2)) <= 1.0
    slice_hu[left_lung | right_lung] = -750.0
    
    # Add pulmonary vasculature texture
    noise = np.random.normal(0, 30, (size, size))
    slice_hu[left_lung | right_lung] += noise[left_lung | right_lung]
    
    # Solitary Pulmonary Nodule (+10 HU soft tissue mass)
    if nodule_row > 0 and nodule_col > 0:
        nodule_dist = np.sqrt((y - nodule_row)**2 + (x - nodule_col)**2)
        nodule_mask = nodule_dist <= nodule_radius
        slice_hu[nodule_mask] = 10.0 + np.random.normal(0, 15, np.sum(nodule_mask))
    else:
        nodule_mask = np.zeros((size, size), dtype=bool)
    
    return slice_hu, nodule_mask

def normalize_lung_window(hu_slice, vmin=-1000.0, vmax=400.0):
    """Applies standard clinical lung windowing to [0.0, 1.0]."""
    clipped = np.clip(hu_slice, vmin, vmax)
    return (clipped - vmin) / (vmax - vmin)

ct_hu, gt_mask = create_synthetic_ct_slice()
ct_norm = normalize_lung_window(ct_hu)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(ct_norm, cmap='gray')
axes[0].plot(316, 366, 'r*', markersize=14, label='Nodule (Row: 366, Col: 316)')
axes[0].set_title('Axial Thoracic CT Slice (Lung Window)')
axes[0].legend(loc='upper right')
axes[0].axis('off')

axes[1].imshow(gt_mask, cmap='magma')
axes[1].set_title(f'Ground Truth Nodule Mask (Pixels: {gt_mask.sum():,})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'Slice Shape: {ct_norm.shape} | Nodule Pixel Fraction: {100 * gt_mask.sum() / gt_mask.size:.4f}%')

## 2. Prompt-Guided Pseudo Ground-Truth Mask Generation (SAM)

Meta's Segment Anything Model (SAM) accepts sparse point prompts. Given the known nodule coordinate (col=316, row=366) and positive foreground label (1), SAM outputs precise morphological boundaries for pseudo ground-truth dataset creation.

In [ ]:
def simulate_sam_point_prompt_segmentation(ct_normalized, prompt_coord, tolerance=0.18):
    """Simulates SAM's two-way attention segmentation guided by a point prompt."""
    row, col = prompt_coord
    target_val = ct_normalized[row, col]
    y, x = np.ogrid[:ct_normalized.shape[0], :ct_normalized.shape[1]]
    spatial_dist = np.sqrt((y - row)**2 + (x - col)**2)
    intensity_diff = np.abs(ct_normalized - target_val)
    
    candidate_mask = (intensity_diff < tolerance) & (spatial_dist < 22)
    labeled, num_features = ndimage.label(candidate_mask)
    prompt_label = labeled[row, col]
    return labeled == prompt_label if prompt_label > 0 else candidate_mask

sam_mask = simulate_sam_point_prompt_segmentation(ct_norm, (366, 316))

overlay = np.stack([ct_norm]*3, axis=-1)
overlay[sam_mask] = [0.85, 0.2, 0.2]  # Red overlay on segmented nodule

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(overlay)
ax.plot(316, 366, 'yo', markersize=8, label='Point Prompt (316, 366)')
ax.set_title('SAM Prompt-Guided Segmentation Overlay')
ax.legend(loc='upper right')
ax.axis('off')
plt.show()

## 3. Segmentation Dataset & DataLoader Construction

We assemble a paired PyTorch `Dataset` producing 2D CT slices normalized to 1 x 256 x 256 and corresponding binary masks 1 x 256 x 256.

In [ ]:
class LunaSegmentationDataset(Dataset):
    """Paired 2D CT slice and binary mask dataset."""
    def __init__(self, num_samples=64, size=256):
        self.num_samples = num_samples
        self.size = size
        self.samples = []
        
        for i in range(num_samples):
            has_nodule = (i % 2 == 0)  # 50% nodule slices, 50% background slices
            if has_nodule:
                r = random.randint(size // 3, 2 * size // 3)
                c = random.choice([random.randint(size // 4, size // 3), random.randint(2 * size // 3, 3 * size // 4)])
                slice_hu, mask = create_synthetic_ct_slice(size=size, nodule_row=r, nodule_col=c, nodule_radius=random.randint(6, 12))
            else:
                slice_hu, mask = create_synthetic_ct_slice(size=size, nodule_row=-100, nodule_col=-100, nodule_radius=0)
            
            slice_norm = normalize_lung_window(slice_hu)
            self.samples.append((slice_norm, mask.astype(np.float32)))
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        slice_norm, mask = self.samples[idx]
        img_tensor = torch.tensor(slice_norm, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return img_tensor, mask_tensor

train_ds = LunaSegmentationDataset(num_samples=48, size=256)
val_ds = LunaSegmentationDataset(num_samples=16, size=256)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

sample_img, sample_mask = train_ds[0]
print(f'Image Tensor: {sample_img.shape} | Mask Tensor: {sample_mask.shape}')

## 4. SegFormer-Inspired Model: Frozen Transformer Encoder + Trainable All-MLP Decoder

We implement an encoder-decoder architecture reflecting SegFormer principles:
- **Hierarchical Encoder (Frozen):** Multi-stage feature extraction.
- **All-MLP Decoder (Trainable):** Projects multi-scale features, upsamples bilinearly, and generates pixel-level nodule logits.

In [ ]:
class ConvTransformerBlock(nn.Module):
    """Efficient self-attention & depthwise mix-FFN block."""
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.BatchNorm2d(channels)
        self.depthwise = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels)
        self.pointwise = nn.Conv2d(channels, channels, kernel_size=1)
        self.act = nn.GELU()
    def forward(self, x):
        res = x
        x = self.act(self.pointwise(self.depthwise(self.norm(x))))
        return res + x

class SegFormerNoduleModel(nn.Module):
    """Lightweight SegFormer-style semantic segmentation model."""
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        # 4-Stage Hierarchical Encoder
        self.stage1 = nn.Sequential(nn.Conv2d(in_channels, 32, 7, stride=2, padding=3), nn.GELU(), ConvTransformerBlock(32))
        self.stage2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.GELU(), ConvTransformerBlock(64))
        self.stage3 = nn.Sequential(nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.GELU(), ConvTransformerBlock(128))
        self.stage4 = nn.Sequential(nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.GELU(), ConvTransformerBlock(256))
        
        # All-MLP Decoder
        embed_dim = 64
        self.linear_c4 = nn.Conv2d(256, embed_dim, 1)
        self.linear_c3 = nn.Conv2d(128, embed_dim, 1)
        self.linear_c2 = nn.Conv2d(64, embed_dim, 1)
        self.linear_c1 = nn.Conv2d(32, embed_dim, 1)
        
        self.classifier = nn.Sequential(
            nn.Conv2d(embed_dim * 4, embed_dim, 1),
            nn.GELU(),
            nn.Conv2d(embed_dim, num_classes, 1)
        )
    
    def forward(self, x):
        h, w = x.shape[-2:]
        c1 = self.stage1(x)   # (B, 32, H/2, W/2)
        c2 = self.stage2(c1)  # (B, 64, H/4, W/4)
        c3 = self.stage3(c2)  # (B, 128, H/8, W/8)
        c4 = self.stage4(c3)  # (B, 256, H/16, W/16)
        
        tgt_size = c1.shape[-2:]
        _c4 = F.interpolate(self.linear_c4(c4), size=tgt_size, mode='bilinear', align_corners=False)
        _c3 = F.interpolate(self.linear_c3(c3), size=tgt_size, mode='bilinear', align_corners=False)
        _c2 = F.interpolate(self.linear_c2(c2), size=tgt_size, mode='bilinear', align_corners=False)
        _c1 = self.linear_c1(c1)
        
        fused = torch.cat([_c4, _c3, _c2, _c1], dim=1)
        logits_half = self.classifier(fused)
        
        logits = F.interpolate(logits_half, size=(h, w), mode='bilinear', align_corners=False)
        return logits

model = SegFormerNoduleModel().to(device)

# Freezing Encoder Stages (Transfer Learning PEFT Strategy)
encoder_modules = [model.stage1, model.stage2, model.stage3, model.stage4]
for stage in encoder_modules:
    for p in stage.parameters():
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Total Model Parameters: {total:,} | Trainable (Decoder Only): {trainable:,} ({100 * trainable / total:.2f}%)')

## 5. Hybrid Loss Function: BCE + Soft Dice Loss

Soft Dice Loss directly maximizes volumetric intersection over union, preventing the network from collapsing to the all-zero background solution.

In [ ]:
class DiceBCELoss(nn.Module):
    """Hybrid loss combining Binary Cross Entropy and Soft Dice Loss."""
    def __init__(self, dice_weight=1.0, smooth=1.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (probs_flat * targets_flat).sum()
        dice_loss = 1.0 - (2.0 * intersection + self.smooth) / (
            probs_flat.sum() + targets_flat.sum() + self.smooth
        )
        
        return bce_loss + self.dice_weight * dice_loss

criterion = DiceBCELoss(dice_weight=1.0)

## 6. Training and Validation Loop

We fine-tune the decoder for 5 epochs using AdamW (lr=5e-4), monitoring the Dice metric on unseen validation CT slices.

In [ ]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4, weight_decay=1e-2)
epochs = 5

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0
    
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item() * imgs.size(0)
        
    avg_train_loss = total_train_loss / len(train_ds)
    
    model.eval()
    total_val_loss = 0.0
    val_dice_scores = []
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = criterion(preds, masks)
            total_val_loss += loss.item() * imgs.size(0)
            
            probs = torch.sigmoid(preds) > 0.5
            intersection = (probs.float() * masks).sum().item()
            dice = (2.0 * intersection) / (probs.float().sum().item() + masks.sum().item() + 1e-6)
            val_dice_scores.append(dice)
            
    avg_val_loss = total_val_loss / len(val_ds)
    mean_val_dice = np.mean(val_dice_scores)
    
    print(f'Epoch [{epoch}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Mean Dice: {mean_val_dice:.4f}')

## 7. Candidate Coordinate Extraction via Connected Components

We run inference on a validation CT slice, threshold the continuous prediction logits, apply connected component labeling, and extract the (row, col) centroid coordinates to propose candidate nodules to the Section 2.6 3D classifier.

In [ ]:
model.eval()
test_img, test_mask = val_ds[0]
test_input = test_img.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(test_input)
    prob_map = torch.sigmoid(logits)[0, 0].cpu().numpy()

bin_mask = prob_map > 0.5
labeled_mask, num_candidates = ndimage.label(bin_mask)

print(f'Detected Candidate Clusters: {num_candidates}')
candidate_centroids = []
if num_candidates > 0:
    centroids = ndimage.center_of_mass(bin_mask, labeled_mask, range(1, num_candidates + 1))
    if isinstance(centroids, tuple):
        centroids = [centroids]
    for idx, (cr, cc) in enumerate(centroids, start=1):
        candidate_centroids.append((cr, cc))
        print(f'  -> Candidate #{idx} Centroid: (Row: {cr:.1f}, Col: {cc:.1f})')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img[0].numpy(), cmap='gray')
axes[0].set_title('Input Axial CT Slice')
axes[0].axis('off')

axes[1].imshow(prob_map, cmap='viridis')
axes[1].set_title('SegFormer Predicted Probability Map')
axes[1].axis('off')

axes[2].imshow(test_img[0].numpy(), cmap='gray')
axes[2].imshow(bin_mask, cmap='Reds', alpha=0.45)
for cr, cc in candidate_centroids:
    axes[2].plot(cc, cr, 'y*', markersize=14, label=f'Candidate ({cr:.0f}, {cc:.0f})')
axes[2].set_title('Proposed Candidate Crops for Ch. 14 Classifier')
if candidate_centroids:
    axes[2].legend(loc='upper right')
axes[2].axis('off')

plt.tight_layout()
plt.show()
print('\n[SUCCESS] Candidate extraction successfully connected Step 2 (Segmentation) with Step 3 (Classification)!')